In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df_clean = pd.read_csv("../data/processed/df_clean.csv")

cols = ["BALANCE", "PURCHASES", "ONEOFF_PURCHASES", "INSTALLMENTS_PURCHASES",
        "CASH_ADVANCE", "CREDIT_LIMIT", "PAYMENTS"]

df_prepare_clustering = df_clean.copy()
for col in cols:
    df_prepare_clustering[col] = np.log1p(df_prepare_clustering[col])

scaler = StandardScaler()
df_prepare_clustering[cols] = scaler.fit_transform(df_prepare_clustering[cols])

In [ ]:
results = []

for n_comp in range(2, 6):
    pca = PCA(n_components=n_comp)
    X_pca = pca.fit_transform(df_prepare_clustering[cols])

    for k in range(2, 8):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_pca)

        sil = silhouette_score(X_pca, labels)
        inertia = kmeans.inertia_

        results.append({
            "n_components": n_comp,
            "k": k,
            "silhouette": sil,
            "inertia": inertia
        })

results_df = pd.DataFrame(results)
results_df.sort_values("silhouette", ascending=False).head(10)